# import libraries

In [2]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import pyproj
import rasterio
from rasterio.crs import CRS
from rasterio.transform import from_origin

# variables

In [4]:
CSV_DIR    = Path('../_data/exports/country_year_exports')       
OUT_DIR    = Path('../_data/exports/roi_tiffs')     

BUFFER_M      = 50.0  
RESOLUTION_M  = 10.0  

CSV_GLOB = "*.csv"     

OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Input CSVs  : {CSV_DIR}")
print(f"Output ROIs : {OUT_DIR}")

Input CSVs  : ../_data/exports/country_year_exports
Output ROIs : ../_data/exports/roi_tiffs


# helper functions

In [5]:
def get_utm_epsg(lon, lat):
    """Return the EPSG code of the UTM zone covering this lon/lat."""
    zone = int((lon + 180) / 6) + 1
    return 32600 + zone if lat >= 0 else 32700 + zone


def create_point_roi_tiff(lat, lon, output_path, buffer_m, resolution_m,):
    """
    Create a small roi.tiff centred on a single lat/lon point.

    The raster is a binary uint8 mask (all pixels = 1).
    CRS is the local UTM zone — required by the Tessera downloader.

    After inference the output will be (H, W, 128).
    YOU ONLY CARE ABOUT THE CENTRE PIXEL: output[H//2, W//2, :]
    """
    epsg    = get_utm_epsg(lon, lat)
    utm_crs = CRS.from_epsg(epsg)

    transformer = pyproj.Transformer.from_crs(
        "EPSG:4326", utm_crs, always_xy=True
    )
    x_utm, y_utm = transformer.transform(lon, lat)

    left   = x_utm - buffer_m
    right  = x_utm + buffer_m
    bottom = y_utm - buffer_m
    top    = y_utm + buffer_m

    width     = int(np.ceil((right - left)   / resolution_m))
    height    = int(np.ceil((top   - bottom) / resolution_m))
    transform = from_origin(left, top, resolution_m, resolution_m)

    # The centre pixel index where YOUR point lives
    cx = width  // 2
    cy = height // 2

    data = np.ones((1, height, width), dtype=np.uint8)

    profile = dict(
        driver="GTiff", dtype="uint8", count=1,
        width=width, height=height,
        crs=utm_crs, transform=transform,
        nodata=0, tiled=True,
        blockxsize=256, blockysize=256,
    )

    output_path.parent.mkdir(parents=True, exist_ok=True)
    with rasterio.open(output_path, "w", **profile) as dst:
        dst.write(data)

    return {"width": width, "height": height, "cx": cx, "cy": cy, "epsg": epsg}


print("Functions defined.")

Functions defined.


# process country_year csv

In [6]:
csv_files = sorted(CSV_DIR.glob(CSV_GLOB))
print(f"Found {len(csv_files)} CSV file(s)")
for f in csv_files:
    print(f"  {f.name}")

Found 18 CSV file(s)
  at_2017.csv
  at_2018.csv
  at_2019.csv
  be_2016.csv
  be_2018.csv
  be_2019.csv
  bg_2016.csv
  bg_2018.csv
  bg_2019.csv
  dk_2016.csv
  dk_2018.csv
  dk_2019.csv
  ee_2016.csv
  ee_2018.csv
  ee_2019.csv
  ie_2017.csv
  ie_2018.csv
  ie_2019.csv


In [17]:
total_created = 0
total_skipped = 0
total_errors  = 0

for csv_path in csv_files:
    file_id = csv_path.stem  

    df = pd.read_csv(csv_path)

    if not {"lat", "long"}.issubset(df.columns):
        print(f"  SKIP — missing 'lat' or 'long' columns")
        continue

    has_country = "country_id" in df.columns
    has_year    = "year"        in df.columns
    has_crop    = "hcat4_name"  in df.columns

    for idx, row in df.iterrows():
        lat = float(row["lat"])
        lon = float(row["long"])

        pt_dir    = OUT_DIR / file_id / f"point_{idx:04d}"
        roi_path  = pt_dir / "roi.tiff"
        meta_path = pt_dir / "meta.json"

        if roi_path.exists():
            total_skipped += 1
            continue

        try:
            grid_info = create_point_roi_tiff(
                lat=lat, lon=lon,
                output_path=roi_path,
                buffer_m=BUFFER_M,
                resolution_m=RESOLUTION_M,
            )

            meta = {
                "file_id":     file_id,
                "row_index":   int(idx),
                "lat":         lat,
                "lon":         lon,
                "country_id":  str(row["country_id"]) if has_country else None,
                "year":        int(row["year"])        if has_year    else None,
                "crop":        str(row["hcat4_name"])  if has_crop    else None,
                "grid_width":  grid_info["width"],
                "grid_height": grid_info["height"],
                "centre_x":    grid_info["cx"],
                "centre_y":    grid_info["cy"],
                "utm_epsg":    grid_info["epsg"],
            }
            with open(meta_path, "w") as f:
                json.dump(meta, f, indent=2)

            total_created += 1

        except Exception as e:
            print(f"  ERROR row {idx} ({lat}, {lon}): {e}")
            total_errors += 1

    print(f"── For {file_id} ── Done ({len(df)} points — crops: {df['hcat4_name'].unique().tolist() if has_crop else 'unknown'}")

print(f"{'='*50}")
print(f"Created : {total_created}")
print(f"Skipped : {total_skipped} (already existed)")
print(f"Errors  : {total_errors}")

── For at_2017 ── Done (2500 points — crops: ['clover', 'maize_corn_popcorn', 'potatoes', 'spring_barley', 'winter_barley']
── For at_2018 ── Done (2500 points — crops: ['clover', 'maize_corn_popcorn', 'potatoes', 'spring_barley', 'winter_barley']
── For at_2019 ── Done (2500 points — crops: ['clover', 'maize_corn_popcorn', 'potatoes', 'spring_barley', 'winter_barley']
── For be_2016 ── Done (2500 points — crops: ['clover', 'maize_corn_popcorn', 'potatoes', 'spring_barley', 'winter_barley']
── For be_2018 ── Done (2500 points — crops: ['clover', 'maize_corn_popcorn', 'potatoes', 'spring_barley', 'winter_barley']
── For be_2019 ── Done (2500 points — crops: ['clover', 'maize_corn_popcorn', 'potatoes', 'spring_barley', 'winter_barley']
── For bg_2016 ── Done (2500 points — crops: ['clover', 'maize_corn_popcorn', 'potatoes', 'spring_barley', 'winter_barley']
── For bg_2018 ── Done (2500 points — crops: ['clover', 'maize_corn_popcorn', 'potatoes', 'spring_barley', 'winter_barley']
── For b

# check everything

In [20]:
# Pick the first roi.tiff created and inspect it
example_roi  = next(OUT_DIR.rglob("roi.tiff"))
example_meta = json.loads(example_roi.parent.joinpath("meta.json").read_text())

with rasterio.open(example_roi) as src:
    print(f"File      : {example_roi}")
    print(f"Size      : {src.width} × {src.height} px")
    print(f"CRS       : {src.crs}")
    print(f"Transform : {src.transform}")
    print(f"Resolution: {src.res} m")
    data = src.read(1)
    print(f"All pixels valid (=1): {data.all()}")

# print(f"\nMeta:")
# print(json.dumps(example_meta, indent=2))

# cx = example_meta["centre_x"]
# cy = example_meta["centre_y"]
# print(f"\nAfter inference: extract embedding[{cy}, {cx}, :] → shape (128,)")

File      : ../_data/exports/roi_tiffs/be_2019/point_0813/roi.tiff
Size      : 10 × 10 px
CRS       : EPSG:32631
Transform : | 10.00, 0.00, 599782.31|
| 0.00,-10.00, 5683111.41|
| 0.00, 0.00, 1.00|
Resolution: (10.0, 10.0) m
All pixels valid (=1): True


In [21]:
# # Optional extra check: verify the centre pixel maps back to your original lat/lon
# import pyproj
# from rasterio.transform import xy

# with rasterio.open(example_roi) as src:
#     # Get the (x, y) coordinates of the centre pixel in UTM
#     x_utm, y_utm = xy(src.transform, cy, cx)  # row, col → x, y
    
#     # Project back to WGS84
#     transformer = pyproj.Transformer.from_crs(
#         src.crs, "EPSG:4326", always_xy=True
#     )
#     lon_check, lat_check = transformer.transform(x_utm, y_utm)

# print(f"Original  lat/lon : {example_meta['lat']:.6f}, {example_meta['lon']:.6f}")
# print(f"Centre px lat/lon : {lat_check:.6f}, {lon_check:.6f}")
# print(f"Difference        : {abs(lat_check - example_meta['lat'])*111000:.1f}m lat, "
#       f"{abs(lon_check - example_meta['lon'])*111000:.1f}m lon")